# 第3章

## 共焦点画像を蛍光チャンネルごとに分けて保存

In [ ]:
import pyvista as pv
from pathlib import Path
import numpy as np
from skimage import io

In [ ]:
# Load a 3D image (numpy array)
img_dir = Path("image/before/wo_vascular_w_drug")

(img_dir / "GFP").mkdir(parents=True, exist_ok=True)
(img_dir / "OE19").mkdir(parents=True, exist_ok=True)

for i1 in [1,2,4,5]:
    for i2 in [1,2,3]:
        if (i1 == 4 and i2 == 3):
            continue
        vol = io.imread(img_dir / f"{i1}_{i2}.tif")
        # 緑色の蛍光チャンネル
        vol_g = vol[:,:,:,0]
        # 赤色の蛍光チャンネル
        vol_r = vol[:,:,:,1]

        io.imsave(img_dir / "GFP" / f"{i1}_{i2}_GFP.tif", vol_g)
        io.imsave(img_dir / "OE19" / f"{i1}_{i2}_OE19.tif", vol_r)

## スフェロイド領域（OE19）のバイナリ化

In [ ]:
from skimage import filters, morphology, measure
import numpy as np
from scipy import ndimage as ndi

# Load a 3D image (numpy array)
img_dir = Path("image/before/w_vascular_w_drug")

(img_dir / "OE19_binary").mkdir(parents=True, exist_ok=True)

for i1 in [1,2,4,5]:
    for i2 in [1,2,3]:
        if (i1 == 4 and i2 == 3):
            continue
        vol_r = io.imread(img_dir / "OE19" / f"{i1}_{i2}_OE19.tif")

        normalized = vol_r.astype(np.float32) 
        normalized = (normalized - normalized.min()) / (normalized.max() - normalized.min())

        smoothed = sitk.GetArrayFromImage(
            sitk.DiscreteGaussian(sitk.GetImageFromArray(normalized), variance=1.0)
        )

        thresh = filters.threshold_otsu(smoothed)
        binary = smoothed > thresh

        binary = morphology.binary_closing(binary, morphology.ball(5))

        binary = ndi.binary_fill_holes(binary)

        min_obj_size = 10000
        binary = morphology.remove_small_objects(binary, min_size=min_obj_size)

        io.imsave(img_dir / "OE19_binary" / f"{i1}_{i2}_OE19_binary.tif", binary)

## DeepVessによって血管内領域（GFP-HUVEC）のセグメンテーションを実行（DeepVessフォルダを参照）

## スフェロイド領域と血管内領域のバイナリ画像からラベル付き統合データの作成

In [ ]:
import numpy as np
from skimage import io
from skimage import filters, morphology, measure

img_dir = Path("image/before/w_vascular_w_drug")

for i1 in [1,2,4,5]:
    for i2 in [1,2,3]:
        if (i1 == 4 and i2 == 3):
            continue
        
        # 1. 画像の読み込み
        GFP_binary = io.imread(img_dir / "GFP_binary" / f"{i1}_{i2}_GFP_binary.tif")
        OE19_binary = io.imread(img_dir / "OE19_binary" / f"{i1}_{i2}_OE19_binary.tif")

        # 2. 背景（すべて3）で初期化した配列を作成
        # 画像のサイズが同じであることを前提としています
        labeled_array = np.full(GFP_binary.shape, 3, dtype=np.uint8)
        
        # 3. がんスフェロイド（OE19）を 2 に設定
        # OE19が 1 の場所を 2 とします
        labeled_array[OE19_binary == 255] = 2
        
        # 4. 血管（GFP）を 1 に設定（最優先）
        # GFPが 255 の場所を 1 とします。
        # これにより、先に設定したスフェロイド（2）の上書きも行われます。
        min_vol = 2700    # これより小さいのはゴミとして削除
        mid_vol = 8000   # これより大きいのは無条件で血管として保持
        min_ratio = 4.0  # 中間サイズのものは、4倍以上細長くないと削除
        vessel_mask = (GFP_binary > 0)
        label_img = measure.label(vessel_mask)
        props = measure.regionprops(label_img)
        refined_vessel = np.zeros_like(vessel_mask)
        
        for prop in props:
            # A. 大きな構造体は無条件でキープ
            if prop.area >= mid_vol:
                refined_vessel[label_img == prop.label] = 1
            
            # B. 中間サイズは「細長い」ものだけキープ
            elif prop.area >= min_vol:
                aspect_ratio = prop.major_axis_length / (prop.minor_axis_length + 1e-7)
                if aspect_ratio >= min_ratio:
                    refined_vessel[label_img == prop.label] = 1
        
        labeled_array[refined_vessel == 1] = 1
        
        # 5. 保存
        io.imsave(img_dir / "labeled_binary" / f"{i1}_{i2}_labeled_binary.tif", labeled_array)

In [ ]:
input_dir = Path("image/before/w_vascular_w_drug")
binary = io.imread(input_dir / "labeled_binary" /"1_1_labeled_binary.tif")
io.imshow(binary[20])

In [ ]:
import pygalmesh

## ラベル付き統合データを立体メッシュ化

In [ ]:
actual_voxel_size = (3.8500, 1.4062, 1.4062)

test_cell_size_map = {
    1: 7.0,  # 血管（細かく）
    2: 25.0,  # スフェロイド（少し粗く）
    3: 200.0   # ゲル（かなり粗く）
}

mesh = pygalmesh.generate_from_array(
    binary, 
    voxel_size=actual_voxel_size,
    max_cell_circumradius=test_cell_size_map, 
    max_facet_distance=1.2, 
    verbose=True
)

## メッシュデータをCOMSOLでimportできるデータ形式に変換、保存

In [ ]:
output_dir = Path("image/before/w_vascular_w_drug/mesh")

output_dir.mkdir(parents=True, exist_ok=True)
mesh.write(output_dir / "1_1.vtk")

In [ ]:
import meshio
mesh = meshio.read(output_dir / "1_1.vtk")

# meshデータの単位がｍなのでμｍになおす
mesh.points = mesh.points * 1e-6

# 保存（COMSOLは .nas を m 単位として読み込みます）
mesh.write(output_dir / "1_1.nas")

# 第4章

## 血管からの距離で重みづけされたスフェロイド領域の体積

In [ ]:
import numpy as np
import csv
from pathlib import Path
from skimage import io
from scipy.ndimage import distance_transform_edt

def calculate_physically_weighted_cancer(label_array, voxel_size=(1.4062, 1.4062, 3.85)):
    """血管(1)からの距離に基づき、がん細胞(2)を指数減衰で重み付けする"""
    vessel_mask = (label_array == 1)
    
    # 血管が存在しない場合の処理
    if not np.any(vessel_mask):
        return np.zeros_like(label_array, dtype=float), None
        
    # 物理的な距離計算
    dist_map = distance_transform_edt(~vessel_mask, sampling=voxel_size)
    
    # 距離200で重み0.01になるよう設計
    sigma = -200.0 / np.log(0.01)
    
    cancer_mask = (label_array == 2)
    weighted_cancer = np.where(cancer_mask, np.exp(-dist_map / sigma), 0.0)
    
    return weighted_cancer, dist_map

# 1. パスの設定
img_dir = Path("image/before/w_vascular_w_drug/labeled_binary")
csv_path = img_dir / "weight_analysis_results.csv"

# 2. 処理対象のファイルリスト取得
image_files = sorted(list(img_dir.glob("*.tif"))) # 拡張子が異なる場合は変更してください

# 3. CSVの作成と書き込み
with open(csv_path, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    # ヘッダーの書き込み
    writer.writerow([
        "file_name", 
        "cancer_voxel_count", 
        "max_weight", 
        "mean_weight_in_cancer", 
        "sum_of_weights"
    ])

    print(f"Processing {len(image_files)} files...")

    for img_path in image_files:
        try:
            # 画像の読み込み
            labeled_img = io.imread(img_path)
            
            # 重み付け計算
            weighted_result, _ = calculate_physically_weighted_cancer(labeled_img)
            
            # 統計量の計算
            cancer_mask = (labeled_img == 2)
            cancer_count = np.sum(cancer_mask)
            
            if cancer_count > 0:
                max_w = np.max(weighted_result)
                mean_w = np.mean(weighted_result[cancer_mask])
                sum_w = np.sum(weighted_result[cancer_mask])
            else:
                max_w = mean_w = sum_w = 0.0

            # CSVに一行追加
            writer.writerow([img_path.name, cancer_count, max_w, mean_w, sum_w])
            print(f"Done: {img_path.name}")

        except Exception as e:
            print(f"Error in {img_path.name}: {e}")

print("-" * 30)
print(f"完了しました。結果は以下に保存されました:\n{csv_path}")

## 重み付き体積を体積で割る

In [ ]:
import pandas as pd
from pathlib import Path

# 1. パスの設定
img_dir = Path("image/before/w_vascular_w_drug/labeled_binary")
csv_path = img_dir / "weight_analysis_results.csv"
output_csv_path = img_dir / "weight_analysis_with_ratio.csv"

# 2. CSVファイルの読み込み
if not csv_path.exists():
    print(f"エラー: {csv_path} が見つかりません。")
else:
    df = pd.read_csv(csv_path)

    # 3. 計算：sum_of_weights を cancer_voxel_count で割る
    # 0除算を防ぐため、ボクセル数が0の場合は0にする処理を含めます
    df['normalized_weight_density'] = df.apply(
        lambda row: row['sum_of_weights'] / row['cancer_voxel_count'] if row['cancer_voxel_count'] > 0 else 0, 
        axis=1
    )

    # 4. 結果の表示（最初の数行）
    print("--- 計算結果（一部） ---")
    print(df[['file_name', 'cancer_voxel_count', 'sum_of_weights', 'normalized_weight_density']].head())

    # 5. 新しいCSVとして保存
    df.to_csv(output_csv_path, index=False, encoding='utf-8')
    print(f"\n計算完了。結果を保存しました: {output_csv_path.name}")